# Inference time comparison between models

# Dependencies

In [ ]:
import importlib.util

REQUIRED_PACKAGES = ["transformers", "torch", "sklearn", "joblib", "pandas", "numpy"]
MISSING_PACKAGES = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]

if MISSING_PACKAGES:
    print("Installing missing packages:", MISSING_PACKAGES)
    !pip install -q transformers torch scikit-learn joblib pandas numpy
else:
    print("All required packages already available.")


# Imports

In [ ]:
import json
import os
import time

import joblib
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# Mount the drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Define Paths and config

In [ ]:
# Edit this if your Drive mount path differs
BASE_DIR = "/content/drive/MyDrive/Capstone"

DATA_DIR = os.path.join(BASE_DIR, "data_v2")
SAVED_DIR = os.path.join(BASE_DIR, "saved")

TRADITIONAL_DIR = os.path.join(SAVED_DIR, "traditional_ml_v2")
DISTILBERT_DIR = os.path.join(SAVED_DIR, "distilbert", "distilbert_best")
MODERNBERT_DIR = os.path.join(SAVED_DIR, "modernbert", "modernbert_best")

TEST_CSV_PATH = os.path.join(DATA_DIR, "test_v2.csv")

EVAL_RESULTS_DIR = os.path.join(BASE_DIR, "eval", "results")
os.makedirs(EVAL_RESULTS_DIR, exist_ok=True)

# Decision threshold — same as baseline notebook (all three at default 0.5 / native boundary)
DECISION_THRESHOLD = 0.5

# max_len per transformer, matching each model's training config
DISTILBERT_MAX_LEN = 512
MODERNBERT_MAX_LEN = 4096

# Timing experiment config
N_SAMPLES = 50
N_WARMUP = 10
TIMING_SEED = 42
DUMMY_WARMUP_TEXT = (
    "This is a throwaway warmup sample used only to trigger one-time "
    "initialization overhead (CUDA context, kernel loading, cuDNN autotune). "
    "It is discarded and never included in any recorded timing."
)

print("BASE_DIR:", BASE_DIR)
print("TEST_CSV_PATH:", TEST_CSV_PATH)
